## Silver Layer Work

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import uuid
from datetime import datetime
from pyspark.sql.functions import *

In [0]:
spark.sql("use catalog novortise_catalog")
spark.sql("create schema if not exists silver")

silver_run_id=str(uuid.uuid4())
print("Current Silver Run Id : ",silver_run_id)

### Step2 - Silver Control Table
This table stores the latest Silver processing state for each entity
It helps us track
- the latest Bronze run already processed by Silver
- the latest Bronze ingestion timestamp already processed
- how many rows were merged in the lates Silver run

In [0]:
%sql
CREATE TABLE IF NOT EXISTS novortise_catalog.silver.processing_control(
  layer string,
  entity_name STRING,
  last_processed_bronze_run_id string,
  last_processed_bronze_ingested_at TIMESTAMP,
  rows_merged bigint,
  silver_run_id STRING,
  updated_at TIMESTAMP
)
using delta

### Step 3 - Helper functions
This cell contains reusable logic for Silver:
upsert_to_silver()
merges cleaned / transformed rows into the Silver target table
- get last_processed_bronze_ingested at reads the Silver watermark upsert_silver_control() updates the Silver control table.
- get_ incremental_bronze() reads only ne Bronze rows that Silver has not processed yet

In [0]:
def upsert_to_silver(df_source,target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable. forName(spark, target_table)
        (dt.alias ("target")
            .merge(df_source.alias ("source"), f"target. (join_key) = source.(join_key")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        )
    else:
        df_source.write. format ("delta"). saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl=(
        spark.table("novortise_catalog.silver.processing_control")
        .filter(
            col("layer")=="silver" & col("entity_name")==entity_name & col("run_status")=="SUCCESS"
        ).orderBy(col("updated_at").desc()).limit(1)
    )
    rows=ctrl.collect()
    if not rows:
        return None
    return rows[0].get("last_processed_bronze_ingested_at")

In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df=spark.createDataFrame(
        [(
            silver,
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "SUCCESS",
            silver_run_id,
            datetime.utcnow(),
        )],
        schema='''
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
        '''
    )

    dt=DeltaTable.forName(spark,"novortise_catalog.silver.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer=s.layer and t.entity_name=s.entity_name")
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id":col("s.last_processed_bronze_run_id"),
                "last_processed_bronze_ingested_at":col("s.last_processed_bronze_ingested_at"),
                "rows_merged":col("s.rows_merged"),
                "run_status":col("s.run_status"),
                "silver_run_id":col("s.silver_run_id"),
                "updated_at":col("s.updated_at")
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df,last_ingested_at

    return bronze_df.filter(col("bronze_ingested_at")>last_ingested_at),last_ingested_at

#### Step 4 Orders incremental processing